# Sistema de agentes, versión 5 — Phi-3.5-mini

Ejecuta las celdas **en orden, de arriba abajo**. El diseño y la regla para aceptar la v5 están
en `PLAN_D3.md`, sección "Versión 5".

| sección | qué hace | cuándo |
|---|---|---|
| 1 a 4 | GPU, clon, autopruebas, Drive, carga del modelo | siempre |
| 5 | la v5 sobre los 223 casos de desarrollo y la regla de aceptación | ahora |
| 6 | bajar los resultados en un zip | al terminar |


## 1 · GPU, dependencias y repositorio
Debe decir `cuda: True` y mostrar una **Tesla T4**. Si no: *Entorno de ejecución* →
*Cambiar tipo de entorno de ejecución* → **T4 GPU**.

`COMMIT` es el commit que se mide; vacío clona la última versión de `main`. Cada fila de
resultados guarda el commit, y el runner no mezcla filas de commits distintos en un archivo.


In [ ]:
COMMIT = ''

%cd /content
!nvidia-smi -L
!pip -q install transformers==5.17.0 accelerate==1.14.0 bitsandbytes==0.50.2
!rm -rf /content/proyecto && git clone -q https://github.com/Luquitas02/genai-inscripcion-ramos /content/proyecto
import subprocess
if COMMIT:
    subprocess.run(['git', '-C', '/content/proyecto', 'checkout', '-q', COMMIT], check=True)
!git -C /content/proyecto log -1 --format='commit %h  %ad  %s'
import torch; print('cuda:', torch.cuda.is_available())
from importlib.metadata import version
for paquete in ('torch', 'transformers', 'accelerate', 'bitsandbytes'):
    print(f'{paquete}=={version(paquete)}')

## 2 · Autopruebas
No usan la GPU. Tienen que terminar en `TODO EN ORDEN`; si alguna falla, no sigas.


In [ ]:
import subprocess, sys, os
os.chdir('/content/proyecto/scripts')
pruebas = (['verificador.py'], ['ficha.py'], ['herramientas.py'], ['runner_d3.py', '--pruebas'],
           ['gen_ablaciones_v5.py', '--check'])
for args in pruebas:
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    ultima = (r.stdout.strip().splitlines() or ['(sin salida)'])[-1]
    print(f'[{"OK  " if r.returncode == 0 else "FALLA"}] {args[0]:22} {ultima[:60]}')
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise SystemExit('*** NO SIGAS: falló ' + args[0] + ' ***')
print('TODO EN ORDEN')

## 3 · Google Drive
**No te saltes esta celda.** Cada conjunto se copia a Drive apenas termina, en
`genai_d3_v5/<commit>`, y lo que ya estaba ahí para este mismo commit se copia al clon para
que el runner retome. Si la sesión se cae a mitad de un conjunto, ese conjunto se repite.


In [ ]:
from google.colab import drive
import shutil
from pathlib import Path
drive.mount('/content/drive')
# Una carpeta por commit: un commit nuevo (otro prompt, otra versión) empieza de cero en vez
# de chocar con lo que se corrió con el anterior, y lo de este commit se retoma.
HASH = subprocess.check_output(['git', '-C', '/content/proyecto', 'rev-parse', '--short=10',
                                'HEAD'], text=True).strip()
RESPALDO = Path('/content/drive/MyDrive/genai_d3_v5') / HASH
PROYECTO = Path('/content/proyecto')
for sub in ('resultados', 'trazas'):
    (RESPALDO / sub).mkdir(parents=True, exist_ok=True)
    (PROYECTO / sub / 'v5').mkdir(parents=True, exist_ok=True)
    for p in (RESPALDO / sub).glob('*.jsonl'):
        shutil.copy(p, PROYECTO / sub / 'v5' / p.name)
        print('recuperado de Drive:', sub, p.name)

def respaldar(*rutas):
    for r in rutas:
        r = Path(r)
        sub = 'resultados' if 'resultados' in r.parts else 'trazas'
        shutil.copy(r, RESPALDO / sub / r.name)
    print('copiado a Drive:', ', '.join(Path(r).name for r in rutas))

print('Drive listo:', RESPALDO)

## 4 · Carga del modelo
Phi-3.5-mini en 4 bits, 256 tokens de salida, generación greedy cortada al cerrar el primer
JSON: lo mismo que la v4. Tarda entre tres y cinco minutos.


In [ ]:
sys.path.insert(0, '/content/proyecto/scripts')
import runner_d3, ver_traza
modelo = runner_d3.cargar_modelo('microsoft/Phi-3.5-mini-instruct')
print('modelo cargado · revisión', getattr(modelo.modelo.config, '_commit_hash', '?'))

## 5 · Desarrollo: la v5 sobre los 223 casos y la regla de aceptación
Los cinco conjuntos de desarrollo: el desarrollo de la v4 (52 + 21) y los 150 de prueba de la
v4, que para la v5 son desarrollo. Unos 30 minutos. Si la sesión se cae, vuelve a correr las
secciones 1 a 4 y esta: retoma donde iba.

Al final, `aceptacion_v5.py` aplica la regla escrita en `PLAN_D3.md` antes de correr y lista
los casos que cambian respecto de la v4. Copia toda la salida de esta celda y pégala en el
chat.


In [ ]:
DESARROLLO = ['casos_desarrollo.jsonl', 'casos_desarrollo_redactados.jsonl',
              'casos.jsonl', 'casos_nuevos.jsonl', 'casos_redactados.jsonl']
datos = PROYECTO / 'datos'
for conjunto in DESARROLLO:
    salida, traza = runner_d3.ejecutar(modelo, 'completo', str(datos / conjunto))
    respaldar(salida, traza)

r = subprocess.run([sys.executable, 'aceptacion_v5.py'], capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])

## 6 · Bajar los resultados
Un zip con `resultados/` y `trazas/` de la v5. También están en Drive, en
`genai_d3_v5/<commit>`.


In [ ]:
from google.colab import files
shutil.make_archive('/content/resultados_v5', 'zip', str(RESPALDO))
files.download('/content/resultados_v5.zip')